## Import modules

In [67]:
%matplotlib inline
import os, sys
base_dir = "../../"
sys.path.extend([f"{base_dir}/models", 
                 f"{base_dir}/utils", 
                 f"{base_dir}/handle_data", 
                 f"{base_dir}/postprocess"])
import numpy as np
import xarray as xr
import pandas as pd
from scores_class import Scores
from evaluation_utils import perform_block_bootstrap_metric as bootstrapping
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt

## Config parameters

In [68]:
cf  = {}
cf['with_snow'] = False
cf['results_basedir'] = "/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/results"
cf['data_dir'] = '/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/dataset/{}'.format(
                        'with_snow' if cf['with_snow'] else 'without_snow')
cf['experiments'] = ["sha_unet_benchmark_t2m", "sha_wgan_benchmark_t2m", "ankit_swinir", "bilinear"] #"deepru_benchmark_t2m" output file is missing
cf['experiment_name'] = "t2m"
cf['ref_experiment'] = "bilinear"
cf['nboots'] = 1000
cf['metrics'] = ["rmse"]

cf['plot_fname'] = "{}_meta_postprocessing.png"


In [69]:
class Config:
    def __init__(self,config_dict : dict):
        self.__dict__.update(config_dict)
config = Config(cf)

In [70]:
rmse_tall_exps= {}
skill_tall_exps = {}

In [71]:
def filename(exp_name : str):
    if exp_name == "bilinear":
        filename = "downscaling_benchmark_t2m_test.nc"
    else:
        filename = "postprocessed_ds_test.nc"
        if not os.path.exists(os.path.join(config.results_basedir,exp_name,filename)):
            experiment_type = exp_name.replace(f"_benchmark_{config.experiment_name}","")
            filename = f"downscaled_{config.experiment_name}_{experiment_type}.nc"
    return filename
    

def input_target_key_determiner(keys):
    forecast_key = list(filter(lambda key: "in" in key or "fcst" in key,keys))[0]
    target_key = list(filter(lambda key: "tar" in key or "ref" in key,keys))[0]
    return forecast_key,target_key



In [72]:
for i, exp in enumerate(config.experiments):
    print(f"Run evaluation for {exp}...")
    ds = xr.open_dataset(
        os.path.join(
            config.data_dir if exp=="bilinear" else os.path.join(config.results_basedir,exp), 
            filename(exp)))
    forecast_key,target_key = input_target_key_determiner(ds.keys())
    score_engine = Scores(ds[forecast_key], ds[target_key], ["rlat", "rlon"])
                    
    rmse_tall_exps[exp] = score_engine(config.metrics[0])

Run evaluation for sha_unet_benchmark_t2m...
Run evaluation for sha_wgan_benchmark_t2m...
Run evaluation for ankit_swinir...
Run evaluation for bilinear...


In [73]:
# get reference
rmse_ref = rmse_tall_exps[config.ref_experiment]

# initialize DataArrays to store results

config.experiments.remove(config.ref_experiment)
nexps = len(config.experiments)

skill_avg = xr.DataArray(np.zeros(nexps), coords={"experiment": config.experiments}, dims="experiment")
skill_boot = xr.DataArray(np.zeros(nexps*config.nboots).reshape(nexps, config.nboots),
                          coords={"experiment": config.experiments, "iboot": np.arange(config.nboots)},
                          dims=["experiment", "iboot"])

for i, exp in enumerate(config.experiments):
    skill_tall_exps[exp] = (rmse_ref - rmse_tall_exps[exp])/rmse_ref
    skill_avg[i], skill_boot[i,:] = skill_tall_exps[exp].mean(), bootstrapping(skill_tall_exps[exp], "time", 120)

In [74]:
def plot_skills(skill_avg, skill_boot, plt_fname, labels=["U-Net (Sha)", "WGAN (Sha)", "DeepRU", "SwinIR"],
                metric="RMSE"):
    fs = 16

    # create figure
    fig, ax = plt.subplots(1, 1)
    # create box-plot
    bp = ax.boxplot(skill_boot.T, labels=labels, patch_artist=True)
    # configure plot
    #ax.set_ylim(0, 1.0)

    ax.set_title("")
    ax.set_ylabel(f"Skill {metric}", fontsize=fs)
    ax.tick_params(axis="both", which="both", direction="out", labelsize=fs-2)

    colors = ['pink', 'lightblue', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)

    for median in bp['medians']:
        median.set_color('black')
        median.set_linewidth(2.)
        
    plt.rcParams['text.usetex'] = True
    ax.text(0.7, 0.05, r"$\overline{RMSE}_{ref}$="+f"{rmse_ref.mean():.2f}K",
        transform=ax.transAxes,
        color='k', fontsize=fs-2)

    fig.savefig(plt_fname, bbox_inches="tight")
    plt.tight_layout()
    fig.savefig(plt_fname)
    plt.close(fig)

In [75]:
plot_skills(skill_avg,
            skill_boot,
            config.plot_fname.format("rmse_plot"),
            labels=['unet','wgan','swinir'])

In [39]:
def create_lines_plot(data: xr.DataArray, data_std: xr.DataArray, model_names: str, metric: dict,
                      plt_fname: str, x_coord: str = "hour", **kwargs):

    # get some plot parameters
    linestyle = kwargs.get("linestyle", ["k-", "b-","o-"])
    err_col = kwargs.get("error_color", ["grey", "blue","green"])
    val_range = kwargs.get("value_range", (0., 3.))
    fs = kwargs.get("fs", 16)
    ref_line = kwargs.get("ref_line", None)
    ref_linestyle = kwargs.get("ref_linestyle", "k--")
    
    fig, (ax) = plt.subplots(1, 1)
    for i, exp in enumerate(data["exp"]):
        ax.plot(data[x_coord].values, data.sel({"exp": exp}).values, linestyle[i],
                label=model_names[i])
        ax.fill_between(data[x_coord].values, data.sel({"exp": exp}).values-data_std.sel({"exp": exp}).values,
                        data.sel({"exp": exp}).values+data_std.sel({"exp": exp}).values, facecolor=err_col[i],
                        alpha=0.2)
    if ref_line is not None:
        nval = np.shape(data[x_coord].values)[0]
        ax.plot(data[x_coord].values, np.full(nval, ref_line), ref_linestyle)
    ax.set_ylim(*val_range)
    # label axis
    ax.set_xlabel("daytime [UTC]", fontsize=fs)
    metric_name, metric_unit = list(metric.keys())[0], list(metric.values())[0]
    ax.set_ylabel(f"{metric_name} T2m [{metric_unit}]", fontsize=fs)
    ax.tick_params(axis="both", which="both", direction="out", labelsize=fs-2)
    ax.legend(fontsize=fs-2, loc="upper right")

    # save plot and close figure
    plt_fname = plt_fname + ".png" if not plt_fname.endswith(".png") else plt_fname
    print(f"Save plot in file '{plt_fname}'")
    plt.tight_layout()
    fig.savefig(plt_fname)
    plt.close(fig)
    
def get_id_from_fname(fname):
    try:
        start_index = fname.find("id") + 2            # Adding 2 to move past "id"
        end_index = fname.find("_", start_index)
        
        exp_id = fname[start_index:end_index]
    except:
        raise ValueError(f"Failed to deduce experiment ID from '{fname}'")
        
    return exp_id

In [40]:
# parameters
config.plt_dir = os.path.join(config.results_basedir, "meta")

varname = "T2m"
year = 2018

In [48]:
# main
os.makedirs(config.plt_dir, exist_ok=True)

fexps = []

for exp in config.experiments:
    fexps.append(os.path.join(config.results_basedir, exp, "metric_files", "eval_rmse_year.csv"))


In [49]:
dims = ["hour", "type"]
coord_dict = {"hour": np.arange(24), "type": ["mean", "std"]}

da_rmse_exps = [xr.DataArray(pd.read_csv(fexp, header=0, index_col=0), dims=dims, coords=coord_dict) for fexp in fexps]


FileNotFoundError: [Errno 2] No such file or directory: '/p/scratch/deepacf/maelstrom/maelstrom_data/ap5/downscaling_benchmark_dataset/benchmark_t2m/results/ankit_swinir/metric_files/eval_rmse_year.csv'

In [ ]:
da_rmse_all = xr.concat(da_rmse_exps, dim= "exp")
da_rmse_all = da_rmse_all.assign_coords({"exp": config.experiments})

In [ ]:
filename = ""
for exp in config.experiments:
    filename+=f"_{exp}"

plt_fname = os.path.join(config.plt_dir, f"eval_rmse_{filename}.png")
create_lines_plot(da_rmse_all.sel({"type": "mean"}), da_rmse_all.sel({"type": "std"}),
                  ['unet','wgan','swinir'], {"RMSE": "K"}, config.plot_fname.format("rmse"))

## Energy Spectra

In [178]:
from evaluation_utils import get_spectrum
import xarray as xr

In [179]:
power_spectrums = []
for i, exp in enumerate(config.experiments):
    print(f"Run evaluation for {exp}...")
    ds = xr.open_dataset(
        os.path.join(
            config.data_dir if exp=="bilinear" else os.path.join(config.results_basedir,exp), 
            filename(exp)))
    forecast_key,target_key = input_target_key_determiner(ds.keys())
    #score_engine = Scores(ds[forecast_key], ds[target_key], ["rlat", "rlon"])
    lonlat_dims = ["rlon","rlat"]
    nlon, nlat = ds[lonlat_dims[0]].size, ds[lonlat_dims[1]].size
    dims = ["wavenumber"]
    coord_dict = {"wavenumber": np.arange(0, np.amin(np.array([int(nlon/2), int(nlat/2)])))}
    ps_exp = get_spectrum(ds[forecast_key],lonlat_dims=lonlat_dims,lcutoff=True,re=6371.)
    power_spectrums.append(xr.DataArray(ps_exp.mean(axis=0),dims=["wavenumber"],coords=coord_dict,name=exp))   
    
    
                    
    #rmse_tall_exps[exp] = score_engine(config.metrics[0])

Run evaluation for sha_unet_benchmark_t2m...
Run evaluation for sha_wgan_benchmark_t2m...
Run evaluation for ankit_swinir...
Run evaluation for bilinear...


KeyboardInterrupt: 

In [111]:
var_unit = "K"

In [127]:
def power_spectra_plot(ds_ps: list[xr.DataArray], var_info: dict, labels: list[str], plt_fname: str, x_coord: str = "wavenumber",**kwargs):

    # get some plot parameters
    linestyle = kwargs.pop("linestyle", "-")
    lw = kwargs.pop("linewidth", 2.)
    cols = kwargs.pop("colors", ["blue","green","orange","red"])
    fs = kwargs.pop("fs", 16)
    fig, (ax) = plt.subplots(1, 1)#, figsize=(12, 8))
    for i, exp in enumerate(ds_ps):
        da = ds_ps[i]
        ax.plot(da[x_coord].values, da.values, linestyle, label=labels[i], lw=lw, color=cols[i], **kwargs)
    ax.set_yscale("log")
    ax.set_title(f"")
    # label axis
    ax.set_xlabel("wavenumber", fontsize=fs)
    var_name, spectrum_unit = list(var_info.keys())[0], list(var_info.values())[0]
    ax.set_ylabel(f"Spectral power {var_name} [{spectrum_unit}]", fontsize=fs)
    ax.tick_params(axis="both", which="both", direction="out", labelsize=fs-2)
    ax.legend(fontsize=fs-2)
    
    plt_fname = plt_fname + ".png" if not plt_fname.endswith(".png") else plt_fname
    print(f"Save plot in file '{plt_fname}'")
    plt.tight_layout()
    fig.savefig(plt_fname)
    plt.close(fig)
   

In [128]:
power_spectra_plot(power_spectrums, {varname: f"{var_unit}**2 m"},['unet','wgan','swinir','bilinear'],config.plot_fname.format("energy_spectra"))

Save plot in file 'energy_spectra_meta_postprocessing.png'


## Model to model comparison

In [5]:
forecast_values = []
target_values = []
for i, exp in enumerate(config.experiments):
    print(f"Run evaluation for {exp}...")
    ds = xr.open_dataset(
        os.path.join(
            config.data_dir if exp=="bilinear" else os.path.join(config.results_basedir,exp), 
            filename(exp)))
    forecast_key,target_key = input_target_key_determiner(ds.keys())
    #score_engine = Scores(ds[forecast_key], ds[target_key], ["rlat", "rlon"])
    forecast_values.append(ds[forecast_key])
    target_values.append(ds[target_key])

Run evaluation for sha_unet_benchmark_t2m...
Run evaluation for sha_wgan_benchmark_t2m...
Run evaluation for ankit_swinir...
Run evaluation for bilinear...


In [6]:
import cartopy.crs as ccrs
from plotting import get_cmap_norm

In [65]:
def create_model_compare_plots(ds_forecasts: list[xr.DataArray],ds_targets: list[xr.DataArray], labels: list[str], plt_fname: str, **kwargs):
    vars2plt = kwargs.pop("vars2plt", None)
    titles = kwargs.pop("titles", None)
    proj_data = kwargs.pop("proj_data", ccrs.RotatedPole(pole_longitude=-162.0, pole_latitude=39.25))
                      
    proj_plot = kwargs.pop("proj_plot", ccrs.PlateCarree())
    dims = kwargs.pop("dims", ["rlat", "rlon"])
    extent = kwargs.pop("extent", [3., 16.5, 43., 51.5])
    fs = kwargs.pop("fs", 16)
    figsize = kwargs.pop("figsize", (12, 12))
    # get levels and colorbars for 'normal' data plots and difference plots
    levels = kwargs.pop("levels", np.arange(-22., 42.1, 2))
    cbar_name = kwargs.pop("cbar_name", "jet")
    levels_diff = kwargs.pop("levels_diff", np.arange(-5.25, 5.01, 0.5))
    cbar_name_diff = kwargs.pop("cbar_name_diff", "PuOr_r")
    cbar_shrink = .8
    
    # auxiliary variables
    lvl, lvl_diff = np.asarray(levels), np.asarray(levels_diff)
    
    lat = ds_forecasts[0]['rlat'].values
    lon = ds_forecasts[0]['rlon'].values
    
    #lat, lon = var_now[dims[0]].values, var_now[dims[1]].values
    # construct array for edges of grid points
    dy, dx = np.round((lat[1] - lat[0]), 4), np.round((lon[1] - lon[0]), 4)
    lat_e, lon_e = np.arange(lat[0]-dy/2, lat[-1]+dy, dy), np.arange(lon[0]-dx/2, lon[-1]+dx, dx)

    # get colormap
    cmap, norm = get_cmap_norm(levels, cbar_name)
    cap_diff, norm_diff = get_cmap_norm(levels_diff, cbar_name_diff)
    
    # create plot objects
    print(len(ds_forecasts))
    fig, axs = plt.subplots(nrows=len(ds_forecasts), ncols=3, figsize=figsize, sharex=True, sharey=True,
                            subplot_kw={"projection": proj_plot})

    for idx,x in enumerate(ds_forecasts):
        print(idx)
        forecast_data = np.squeeze(np.mean(ds_forecasts[idx].values,axis=0))
        plot_fcst_data = axs[idx,0].pcolormesh(lon_e, lat_e, forecast_data, cmap=cmap, norm=norm, transform=proj_data,
                                        **kwargs)
        #axs[idx,0].set_title(titles[idx])
        target_data = np.squeeze(np.mean(ds_targets[idx].values,axis=0))
        plot_tar_data = axs[idx,1].pcolormesh(lon_e, lat_e, target_data, cmap=cmap, norm=norm, transform=proj_data,
                                        **kwargs)
        #axs[idx,1].set_title(titles[idx])
        
        print(ds_forecasts[idx].shape)
        print(ds_targets[idx].shape)
        difference_data = np.squeeze(np.mean(ds_forecasts[idx]-ds_targets[idx],axis=0))
        print(difference_data.shape)
        plot_diff = axs[idx,2].pcolormesh(lon_e, lat_e, difference_data.values, cmap=cmap, norm=norm, transform=proj_data,
                                        **kwargs)
        #axs[idx,2].pcolormesh(lon_e, lat_e, difference_data, cmap=cmap, norm=norm, transform=proj_data,
        #                                **kwargs)
        
    cbar = fig.colorbar(plot_fcst_data, ax=axs[0:2], orientation="vertical", shrink=cbar_shrink,
                        pad=.02, ticks=lvl[1::2], fraction=0.02)
    cbar.ax.tick_params(labelsize=fs-2)
    
    
    cbar_tar = fig.colorbar(plot_tar_data, ax=axs[0:2], orientation="vertical", shrink=cbar_shrink,
                        pad=.02, ticks=lvl[1::2], fraction=0.02)
    cbar_tar.ax.tick_params(labelsize=fs-2)
    
#     cbar_diff = fig.colorbar(plot_diff, ax=axs[-1], orientation="vertical", shrink=1.5,
#                              pad=.04, ticks=lvl_diff[1::2], fraction=0.02)
#     cbar_diff.ax.tick_params(labelsize=fs-2)
    
    # add colorbars
    cbar = fig.colorbar(plot_fcst_data, ax=axs[0:2], orientation="vertical", shrink=cbar_shrink,
                        pad=.02, ticks=lvl[1::2], fraction=0.02)
    cbar.ax.tick_params(labelsize=fs-2)
    cbar_tar = fig.colorbar(plot_tar_data, ax=axs[0:2], orientation="vertical", shrink=cbar_shrink,
                        pad=.02, ticks=lvl[1::2], fraction=0.02)
    cbar_tar.ax.tick_params(labelsize=fs-2)
    
#     cbar_diff = fig.colorbar(plt_diff, ax=axs[-1], orientation="vertical", shrink=1.5,
#                              pad=.04, ticks=lvl_diff[1::2], fraction=0.02)
#     cbar_diff.ax.tick_params(labelsize=fs-2)
    
    cbar = fig.colorbar(plot_diff, ax=axs[0:2], orientation="vertical", shrink=cbar_shrink,
                        pad=.02, ticks=lvl[1::2], fraction=0.02)
    cbar.ax.tick_params(labelsize=fs-2)
    
    plt_fname = plt_fname + ".png" if not plt_fname.endswith(".png") else plt_fname
    fig.savefig(plt_fname, bbox_inches="tight", dpi=300)
    plt.close(fig)

In [66]:
create_model_compare_plots(forecast_values,target_values,['unet','wgan','swinir','bilinear'],config.plot_fname.format("inter_model"))

4
0
(8748, 128, 144)
(8748, 128, 144)
(128, 144)
1
(8748, 128, 144)
(8748, 128, 144)
(128, 144)
2
(8748, 128, 144)
(8748, 128, 144)
(128, 144)
3
(8748, 128, 144)
(8748, 128, 144)
(128, 144)


In [ ]:
class MetaPostProcessor(Object):
    def __init__(self,season_type,metric,plot_type,models,use_score**kwargs):
        assert(season_type in ['year','DJF','MAM','JJA','SON'])
        assert(metric in ['rmse','grad_amplitude','bias'])
        assert(plot_type in ['box_plot','daytime_line_plot','energy_spectra','case_study_plots'])
        self.season = season
        self.metric_plot = metric_plot
        self.use_score = use_score
        #if metric_type == 'model_difference' and use_score:
        #    raise ValueError(f"metric type {metric_type} and use_score {use_score} is not available. Try different combination")
            
        self.score_function = 
            
    
    def add_config(self, config : Config):
        self.config = config
        
    def plot(self):
        pass